# CICIDS-2017 EDA & Preprocessing for CM-MTD Architecture

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold


import glob
import os

# Set display options for comprehensive EDA
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings('ignore')

: 

In [ ]:
dataset_path = './dataset/cicids2017' 

# Use glob to grab all .csv files in the directory
all_files = glob.glob(os.path.join(dataset_path, "*.csv"))

print(f"Found {len(all_files)} files. Beginning merge...")

# List to hold the individual DataFrames
df_list = []

for file in all_files:
    # Read each file
    print(f"Reading {os.path.basename(file)}...")
    df = pd.read_csv(file)
    
    # CICIDS-2017 is notorious for having leading/trailing spaces in column names.
    # Stripping them here prevents misalignment when concatenating.
    df.columns = df.columns.str.strip()
    
    df_list.append(df)

# Concatenate all DataFrames into one massive DataFrame
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

print(f"\nMerge complete! Total rows: {combined_df.shape[0]}, Total columns: {combined_df.shape[1]}")

# Save the combined DataFrame to a single CSV file
output_filename = 'cicids2017_combined.csv'
combined_df.to_csv(output_filename, index=False)

print(f"Successfully saved as {output_filename}")

In [ ]:
# Load and Inspect
# Replace with your local path to the dataset
file_path = 'dataset/cicids2017/cicids2017_combined.csv' 

try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print("Dataset not found. Please ensure the path is correct.")
    # Creating a dummy dataframe to allow the notebook to run for demonstration
    # In practice, load the actual CICIDS2017 CSVs here.
    pass 

# Strip leading/trailing whitespaces from column names (Common issue in CICIDS-2017)
df.columns = df.columns.str.strip()

# Display basic info
display(df.info())
display(df.describe())

# Check the distribution of the raw labels
plt.figure(figsize=(12, 6))
sns.countplot(y=df['Label'], order=df['Label'].value_counts().index)
plt.title('Raw Label Distribution in CICIDS-2017')
plt.xscale('log') # Log scale due to heavy class imbalance
plt.show()

In [ ]:
# Map and Filter Target Classes
# Define mapping dictionary
label_mapping = {
    'BENIGN': 'Benign',
    'DoS Hulk': 'DoS/DDoS',
    'DoS GoldenEye': 'DoS/DDoS',
    'DoS slowloris': 'DoS/DDoS',
    'DoS Slowhttptest': 'DoS/DDoS',
    'DDoS': 'DoS/DDoS',
    'Infiltration': 'Infiltration'
}

# Apply mapping; unmapped labels become NaN
df['Target'] = df['Label'].map(label_mapping)

# Drop rows that do not fall into the three categories evaluated in the paper
df = df.dropna(subset=['Target'])
df = df.drop(columns=['Label'])

print("Filtered Label Distribution:")
print(df['Target'].value_counts())

# Encode the target classes for the LSTM/Softmax layer
le = LabelEncoder()
df['Target_Encoded'] = le.fit_transform(df['Target'])
print("\nClass Mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Data Cleaning (Handling Inf and NaN)
# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check missing values
missing_percentages = (df.isnull().sum() / len(df)) * 100
print("Missing values per feature (%):\n", missing_percentages[missing_percentages > 0])

# Strategy: Drop rows with NaN (Best practice for CICIDS-2017 since NaNs are usually < 1%)
df.dropna(inplace=True)
print(f"\nShape after dropping NaNs: {df.shape}")


## Feature Selection & Dimensionality Reduction
The paper mentions mapping events into a "low dimensional space"[cite: 326]. High-dimensional, highly correlated features degrade DRL convergence and cause state-space explosion. 

We will:
1. Drop metadata/string columns not usable by the network (e.g., IPs, though in a real SDN controller, IPs are used for routing, the neural network requires numerical features).
2. Remove Zero-Variance features.
3. Remove highly correlated features (Pearson correlation > 0.90) to reduce redundancy.

In [ ]:
# Apply Feature Selection
# Separate features and target
X = df.drop(columns=['Target', 'Target_Encoded'])
y = df['Target_Encoded']

# 1. Drop non-numeric features if any remain (e.g., Flow ID, Source IP, Destination IP)
# *Note: The paper notes node ID and timestamp are used. If retaining timestamps, 
# convert them to UNIX epoch or cyclical time features (sin/cos).*
cols_to_drop = X.select_dtypes(include=['object']).columns
X = X.drop(columns=cols_to_drop)

# 2. Remove constant features (Zero Variance)
selector = VarianceThreshold(threshold=0.0)
selector.fit(X)
X_var = X.loc[:, selector.get_support()]
print(f"Features removed due to zero variance: {X.shape[1] - X_var.shape[1]}")

# 3. Remove highly correlated features
corr_matrix = X_var.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper_tri.columns if any(upper_tri[column] > 0.90)]

X_final = X_var.drop(columns=to_drop_corr)
print(f"Features removed due to high correlation (>0.90): {len(to_drop_corr)}")
print(f"Final feature count for environment state: {X_final.shape[1]}")

In [ ]:
# Split and Scale
# 80/20 Train-Test Split as specified by the paper
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

# Scale the features
scaler = StandardScaler()

# Fit on training data ONLY to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling if passing to a custom Gym environment
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_final.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_final.columns)

display(X_train_scaled_df.head())

In [ ]:
# Save Data
# Save the preprocessed arrays for the DRL pipeline
np.save('X_train_env_state.npy', X_train_scaled)
np.save('X_test_env_state.npy', X_test_scaled)
np.save('y_train_env_state.npy', y_train)
np.save('y_test_env_state.npy', y_test)

print("Preprocessing complete. Data saved for LSTM/HDRL ingestion.")